# 📘 Notebook 06 · DeFi + 链上数据分析

> 这一节是最有意思的部分：**我们用 Python 复刻 Uniswap V2**，搞懂 AMM 的数学。然后学怎么读链上数据找机会。

**学完你会：**

- 理解 AMM 的恒定乘积公式、滑点、无常损失
- 用 Python 实现一个 mini Uniswap V2，包括 swap、add/remove liquidity
- 知道 Aave / Compound 的借贷利率怎么算
- 用 Web3.py 直接查 Uniswap 主网池子状态
- 知道 Dune Analytics、The Graph 怎么用

**预计时间：10-12 小时**

**前置**：Notebook 04-05。

## 1. AMM 是什么，为什么能颠覆传统交易所

### 1.1 传统交易所 vs DEX

**传统交易所**（如 Nasdaq、币安现货）用**订单簿**（order book）撮合：

```
买单：              卖单：
100 USDT @ $2010    100 USDT @ $2015
50 USDT @ $2009     200 USDT @ $2020
30 USDT @ $2005     ...
...

成交：买方价 >= 卖方价
```

**问题**：链上跑订单簿太贵。每个挂单都要 SSTORE，每秒处理几千笔在 EVM 上是天文数字 Gas。

### 1.2 AMM 的天才想法

不要订单簿了！**用一个"水池"做市**：

- 池子里有两种代币，比如 ETH 和 USDC
- 任何人可以：往池子加流动性（添加两种币）、从池子换币
- 价格由**池子里两种币的比例**决定，没有挂单

**核心公式（Uniswap V2 恒定乘积）：**

$$x \cdot y = k$$

- $x$：池子里 token A 的数量
- $y$：池子里 token B 的数量
- $k$：常数

换币时：你给池子加 $\Delta x$ 的 A，从池子拿走 $\Delta y$ 的 B，必须保持 $(x + \Delta x)(y - \Delta y) = k$。

这意味着**买得越多、价格越差**——自然形成了滑点。

### 1.3 为什么这是革命

- ✅ 完全链上，每笔交易都是一次合约调用
- ✅ 任何人都能加流动性（LP），不需要做市商
- ✅ 价格自动发现，无需挂单
- ✅ 流动性可组合（一个 LP token 可以作为另一个协议的抵押品）

Uniswap V2 合约 < 300 行，承载了 $1 万亿+ 累计交易量。

## 2. 用 Python 实现一个完整 AMM

下面我们写一个 `AMMPool` 类，**支持 Uniswap V2 的所有核心操作**：
- 添加流动性
- 移除流动性
- swap（带手续费、滑点）
- LP 代币管理

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False


class AMMPool:
    """Uniswap V2 风格的恒定乘积 AMM"""

    def __init__(self, name_a='ETH', name_b='USDC', fee_rate=0.003):
        self.name_a = name_a
        self.name_b = name_b
        self.reserve_a = 0.0
        self.reserve_b = 0.0
        self.total_lp = 0.0          # LP 代币总供应
        self.lp_balances = {}        # LP 持有者 -> 数量
        self.fee_rate = fee_rate      # 默认 0.3%（Uniswap V2 标准）
        self.fees_collected_a = 0
        self.fees_collected_b = 0

    @property
    def k(self):
        return self.reserve_a * self.reserve_b

    @property
    def price(self):
        """1 单位 A 值多少 B（如 1 ETH = ? USDC）"""
        return self.reserve_b / self.reserve_a if self.reserve_a > 0 else 0

    def add_liquidity(self, lp_addr: str, amount_a: float, amount_b: float):
        """
        添加流动性
        - 首次添加：自动定义初始比例（也就是初始价格）
        - 之后：必须按当前池子比例添加，否则按更少的那边匹配
        - 返回：mint 的 LP 代币数量
        """
        if self.total_lp == 0:
            # 初始流动性：LP 数 = sqrt(a*b)
            lp = (amount_a * amount_b) ** 0.5
            self.reserve_a = amount_a
            self.reserve_b = amount_b
        else:
            # 后续：按当前比例
            ratio_a = amount_a / self.reserve_a
            ratio_b = amount_b / self.reserve_b
            ratio = min(ratio_a, ratio_b)
            actual_a = self.reserve_a * ratio
            actual_b = self.reserve_b * ratio
            lp = self.total_lp * ratio
            self.reserve_a += actual_a
            self.reserve_b += actual_b
            # 多出来的部分按理论上要退还，简化起见不退
            amount_a, amount_b = actual_a, actual_b

        self.total_lp += lp
        self.lp_balances[lp_addr] = self.lp_balances.get(lp_addr, 0) + lp
        return lp, amount_a, amount_b

    def remove_liquidity(self, lp_addr: str, lp_amount: float):
        """移除流动性：销毁 LP 换回两种代币"""
        if lp_amount > self.lp_balances.get(lp_addr, 0):
            raise ValueError('LP 余额不足')
        share = lp_amount / self.total_lp
        amount_a = self.reserve_a * share
        amount_b = self.reserve_b * share
        self.reserve_a -= amount_a
        self.reserve_b -= amount_b
        self.total_lp -= lp_amount
        self.lp_balances[lp_addr] -= lp_amount
        return amount_a, amount_b

    def get_amount_out(self, amount_in: float, token_in: str):
        """给定输入数量，算输出多少（不实际执行）"""
        if token_in == self.name_a:
            x_in, x_out = self.reserve_a, self.reserve_b
        else:
            x_in, x_out = self.reserve_b, self.reserve_a
        # 扣手续费
        amount_in_with_fee = amount_in * (1 - self.fee_rate)
        # k = x_in * x_out, 新 x_in = x_in + amount_in_with_fee
        # 新 x_out = k / 新 x_in
        amount_out = x_out - (x_in * x_out) / (x_in + amount_in_with_fee)
        return amount_out

    def swap(self, amount_in: float, token_in: str):
        """执行 swap"""
        amount_out = self.get_amount_out(amount_in, token_in)
        if token_in == self.name_a:
            self.reserve_a += amount_in
            self.reserve_b -= amount_out
            self.fees_collected_a += amount_in * self.fee_rate
        else:
            self.reserve_b += amount_in
            self.reserve_a -= amount_out
            self.fees_collected_b += amount_in * self.fee_rate
        return amount_out

    def __repr__(self):
        return (f'AMMPool({self.name_a}={self.reserve_a:.4f}, '
                f'{self.name_b}={self.reserve_b:.4f}, '
                f'price={self.price:.4f}, k={self.k:.2f})')

# ---------- 演示 ----------
pool = AMMPool('ETH', 'USDC', fee_rate=0.003)

# Alice 是初始流动性提供者：放 100 ETH + 200,000 USDC
# 这同时定义了初始价格 = 200,000 / 100 = 2000 USDC/ETH
lp_alice, _, _ = pool.add_liquidity('alice', 100, 200_000)
print(f'初始化池子: {pool}')
print(f'Alice 拿到 {lp_alice:.2f} LP 代币')

In [ ]:
# Bob 来 swap：用 1000 USDC 买 ETH
amount_in = 1000
amount_out = pool.swap(amount_in, 'USDC')

print(f'\nBob 用 {amount_in} USDC 换到 {amount_out:.6f} ETH')
print(f'平均成交价: {amount_in / amount_out:.4f} USDC/ETH')
print(f'交易前池子价格: 2000')
print(f'交易后池子价格: {pool.price:.4f}')
print(f'\n池子状态: {pool}')

**注意几个细节：**

1. Bob 实际成交价 ~2010 USDC/ETH，**比交易前价格 2000 高**——这就是**滑点**
2. 交易后池子价格也变了——大单会"推动"价格
3. 0.3% 手续费被池子吞了，归 LP（Alice）所有

### 滑点公式

给定输入 $\Delta x$，输出 $\Delta y$（扣手续费后）：

$$\Delta y = y - \frac{x \cdot y}{x + (1-f)\Delta x}$$

其中 $f$ 是手续费率。

价格冲击 = $1 - \frac{\Delta y / \Delta x}{y / x}$

In [ ]:
# 滑点 vs 交易量的关系
pool2 = AMMPool('ETH', 'USDC', fee_rate=0.003)
pool2.add_liquidity('alice', 100, 200_000)

mid_price = 200_000 / 100   # 初始价格

trade_sizes = np.linspace(100, 50_000, 100)
slippages = []
exec_prices = []

for size in trade_sizes:
    out = pool2.get_amount_out(size, 'USDC')
    exec_price = size / out      # USDC/ETH
    slippage = (exec_price - mid_price) / mid_price
    slippages.append(slippage)
    exec_prices.append(exec_price)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(trade_sizes, np.array(slippages) * 100, color='darkred')
axes[0].set_xlabel('交易金额 (USDC)'); axes[0].set_ylabel('滑点 (%)')
axes[0].set_title('交易量 vs 滑点')
axes[0].grid(alpha=0.3)

axes[1].plot(trade_sizes, exec_prices, color='steelblue', label='成交价')
axes[1].axhline(mid_price, color='gray', linestyle='--', alpha=0.6, label='中间价')
axes[1].set_xlabel('交易金额 (USDC)'); axes[1].set_ylabel('成交价 (USDC/ETH)')
axes[1].set_title('交易量 vs 成交价')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 池子越大滑点越小，所以"深度"决定了 AMM 的可用性
print(f'1000 USDC 换 ETH 滑点 : {(slippages[2])*100:.3f}%')
print(f'10000 USDC 换 ETH 滑点 : {(slippages[20])*100:.3f}%')
print(f'50000 USDC 换 ETH 滑点 : {(slippages[-1])*100:.3f}%')

## 3. 无常损失（Impermanent Loss）

**LP 不是稳赚不赔的。** 价格变动会导致 LP 持有的总价值低于"如果你只是 hold 这两个币"的情况。这个差额叫**无常损失**。

### 3.1 直觉

你放 100 ETH + 200k USDC 入池，初始价格 2000。

如果 ETH 涨到 4000（外部市场），套利者会从池子买便宜的 ETH，**直到池子价格也变成 4000**。

**等池子重新平衡后，你的份额会变成：** ETH 更少 + USDC 更多。

总价值通常**小于"如果你只是拿着 100 ETH + 200k USDC 不动"**。

### 3.2 数学公式

设价格变化倍数 $p = P_1 / P_0$。无常损失（占初始价值的比例）：

$$IL(p) = \frac{2\sqrt{p}}{1 + p} - 1$$

| 价格变化 | 无常损失 |
|---|---|
| 1.0x（不变）| 0% |
| 1.25x | -0.6% |
| 1.5x | -2.0% |
| 2x | -5.7% |
| 3x | -13.4% |
| 4x | -20.0% |
| 5x | -25.5% |
| 10x | -42.6% |

**价格涨跌都会有损失**（涨 2x 和跌 2x 的损失相同）。

In [ ]:
def impermanent_loss(price_ratio: float) -> float:
    """价格变 price_ratio 倍后的无常损失"""
    return 2 * np.sqrt(price_ratio) / (1 + price_ratio) - 1

# 画无常损失曲线
ratios = np.linspace(0.1, 10, 200)
il = [impermanent_loss(r) for r in ratios]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ratios, np.array(il) * 100, color='darkred')
ax.axhline(0, color='gray', alpha=0.5)
ax.axvline(1, color='gray', alpha=0.5, linestyle='--')
ax.set_xlabel('价格变化倍数 (P1/P0)')
ax.set_ylabel('无常损失 (%)')
ax.set_title('无常损失 vs 价格变化（对称）')
ax.grid(alpha=0.3)
# 标几个关键点
for r in [0.5, 1, 2, 5]:
    ax.scatter(r, impermanent_loss(r)*100, color='red', zorder=5)
    ax.annotate(f'{r}x: {impermanent_loss(r)*100:.1f}%',
                (r, impermanent_loss(r)*100), xytext=(10, -10),
                textcoords='offset points', fontsize=9)
plt.tight_layout(); plt.show()

**LP 的真实收益 = 手续费收入 - 无常损失**

- 高波动币对：手续费多，但无常损失也大
- 稳定币对（USDC/USDT）：无常损失几乎为 0（用 Curve 的"稳定币 AMM"会更优）
- 波动方向 + 持续时间也重要（"无常"= 价格回到原点损失也消失）

**Uniswap V3 引入"集中流动性"**，让 LP 可以选择价格区间，资金效率提高 100x，但无常损失也更敏感。

### 🎯 练习

三个练习：

1. 模拟一个场景：池子价格从 2000 走到 4000，LP 持有 1 个月，每天产生 1% 交易量。算 LP 最终的真实收益（含手续费 - IL）
2. 改写 AMMPool 让它支持 "k = c1·x + c2·y"（恒定和模型 / StableSwap 简化版），对比稳定币对的滑点
3. 写一个 `simulate_lp_pnl(price_path, daily_volume)` 函数，输入价格路径和日成交量，输出 LP 的累计盈亏曲线

In [ ]:
# 在这里写你的答案


## 4. Uniswap V3：集中流动性

V2 的问题：**LP 资金大部分在没用的价格区间**。

ETH/USDC 池子里，ETH 不会跌到 0，也不会涨到无穷，可绝大多数 LP 资金"覆盖"了 [0, ∞] 的全价格范围。

### 4.1 V3 的解决方案

LP 选择一个**价格区间** $[P_a, P_b]$ 提供流动性。只在这个区间内：
- 你的资金被使用、赚手续费
- 出了区间，你的资金全部变成"那边的币"（如价格涨出区间，全部变成 USDC）

### 4.2 数学

V2: $x \cdot y = k$
V3 (在某个 tick 区间内): $(x + a)(y + b) = L^2$

其中 $L$ 是流动性参数，$a, b$ 与价格区间端点相关。

实际中 V3 把价格离散成"tick"（每个 tick = 0.01% 价差），LP 在 tick 区间内提供 L。**这套数学比 V2 复杂多了，但基本原理就是"虚拟储备"概念**。

### 4.3 资金效率提升

如果 ETH/USDC 池子里 90% 的交易发生在 [1800, 2200] 区间：
- V2 LP：100k 资金被均摊到 [0, ∞]，效率低
- V3 LP：把 100k 集中在 [1800, 2200]，**资金效率提升 ~50x**

**代价**：LP 需要主动管理（价格出区间要重新调整），无常损失更显性。

> 学 V3 数学的最好资源是 [Uniswap V3 白皮书](https://uniswap.org/whitepaper-v3.pdf) + 几个解读博客。本节不展开。

## 5. 借贷协议：Aave / Compound 数学

DeFi 的另一个核心赛道是**借贷**。原理简单到一句话：

> 用户存币赚利息，借币付利息。利率由"利用率"动态决定。

### 5.1 核心公式

**利用率（utilization rate）：**

$$U = \frac{\text{Total Borrowed}}{\text{Total Supplied}}$$

**借款利率（分段线性，Aave 风格）：**

$$
r_{borrow}(U) = \begin{cases}
r_0 + U \cdot \frac{r_{slope1}}{U^*} & \text{if } U \le U^* \\
r_0 + r_{slope1} + (U - U^*) \cdot \frac{r_{slope2}}{1 - U^*} & \text{if } U > U^*
\end{cases}
$$

- $U^*$：最优利用率（通常 80%）
- $U < U^*$ 时：利率温和增长
- $U > U^*$ 时：利率陡升（鼓励还款 / 提取）

**存款利率：**

$$r_{supply} = r_{borrow} \cdot U \cdot (1 - \text{reserveFactor})$$

意思是：存款利率 = 借款利率 × 利用率 × (1 - 协议抽成)。

In [ ]:
def borrow_rate(U, r0=0.0, r1=0.04, r2=0.75, U_star=0.8):
    """Aave 风格借款利率曲线"""
    if U <= U_star:
        return r0 + U * (r1 / U_star)
    else:
        return r0 + r1 + (U - U_star) * (r2 / (1 - U_star))

def supply_rate(U, r0=0.0, r1=0.04, r2=0.75, U_star=0.8, reserve_factor=0.1):
    return borrow_rate(U, r0, r1, r2, U_star) * U * (1 - reserve_factor)

# 画利率曲线
U_range = np.linspace(0, 1, 200)
borrow = [borrow_rate(u) for u in U_range]
supply = [supply_rate(u) for u in U_range]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(U_range * 100, np.array(borrow) * 100, color='darkred', label='借款利率')
ax.plot(U_range * 100, np.array(supply) * 100, color='darkgreen', label='存款利率')
ax.axvline(80, color='gray', linestyle='--', alpha=0.5, label='最优利用率 U*=80%')
ax.set_xlabel('利用率 (%)')
ax.set_ylabel('年化利率 (%)')
ax.set_title('Aave 风格利率曲线')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('利用率 50% 时借款利率:', f'{borrow_rate(0.5)*100:.2f}%')
print('利用率 80% 时借款利率:', f'{borrow_rate(0.8)*100:.2f}%')
print('利用率 95% 时借款利率:', f'{borrow_rate(0.95)*100:.2f}%   # 陡升')

### 5.2 清算机制

**借款必须超额抵押。** 你存 100 USDC 想借 ETH，最多借不到 100 USDC 等值（如能借 75%）。

**清算条件：**

$$\text{Health Factor} = \frac{\text{Collateral Value} \cdot \text{Liquidation Threshold}}{\text{Borrowed Value}}$$

HF < 1 → 任何人都可以清算这个借款人，**清算者把借款人的债还掉，拿走他的抵押品并获得 5-10% 折扣**。

### 5.3 闪电贷（Flash Loan）

DeFi 独有的金融创新：**一笔交易内借出 → 用 → 还回，否则整笔回滚**。

```solidity
// 伪代码
function flashLoan(uint amount) {
    transfer(borrower, amount);          // 借出
    borrower.execute();                   // 执行任意操作
    require(balanceOf(this) >= amount + fee, "未还款");  // 强制还款
}
```

应用：
- **套利**（Notebook 07 会展开）：发现 DEX A 比 DEX B 便宜，闪电贷 1000 ETH 套利，秒还
- **抵押品迁移**：把 Compound 上的债务搬到 Aave，闪电贷做中间杠杆
- **清算**：用闪电贷清算别人，赚清算奖励

**闪电贷不是漏洞，是 DeFi 的合法特性。但它也被用来发起攻击**（操纵预言机、操纵 AMM 价格）。

## 6. 直接读 Uniswap V2 主网池子数据

理论说够了。我们用 web3.py **直接读以太坊主网的 Uniswap V2 ETH/USDC 池子**，看看真实状态。

In [ ]:
# 需要一个 RPC 节点。免费选项：
# - Alchemy 注册 → 拿 API key
# - 公共 RPC: https://eth.llamarpc.com（免费，限流）
#
# pip install web3

DEMO_CODE = '''
from web3 import Web3

# 用免费公共 RPC（生产建议自建或 Alchemy）
w3 = Web3(Web3.HTTPProvider("https://eth.llamarpc.com"))
print("已连接:", w3.is_connected())

# Uniswap V2 ETH/USDC 池子地址（主网）
PAIR_ADDR = "0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"

# Uniswap V2 Pair 的最小 ABI（只要 getReserves 和 token 函数）
ABI = [
  {"inputs":[],"name":"getReserves",
   "outputs":[{"name":"_reserve0","type":"uint112"},
              {"name":"_reserve1","type":"uint112"},
              {"name":"_blockTimestampLast","type":"uint32"}],
   "stateMutability":"view","type":"function"},
  {"inputs":[],"name":"token0",
   "outputs":[{"name":"","type":"address"}],
   "stateMutability":"view","type":"function"},
  {"inputs":[],"name":"token1",
   "outputs":[{"name":"","type":"address"}],
   "stateMutability":"view","type":"function"}
]

pair = w3.eth.contract(address=PAIR_ADDR, abi=ABI)

t0 = pair.functions.token0().call()
t1 = pair.functions.token1().call()
r0, r1, ts = pair.functions.getReserves().call()

# USDC 是 6 位小数，WETH 是 18 位小数
# token0 是 USDC (0xA0b86991...)
# token1 是 WETH
usdc_reserve = r0 / 10**6
weth_reserve = r1 / 10**18
print(f"池子 USDC : {usdc_reserve:,.2f}")
print(f"池子 WETH : {weth_reserve:,.4f}")
print(f"隐含价格 : {usdc_reserve / weth_reserve:.2f} USDC/WETH")
'''
print(DEMO_CODE)
print('\n上面代码可以直接放到本机跑（装好 web3 + 联网）。')
print('用 EthereumTester 就跑不了——它没有主网状态。')

### 6.1 用闪电贷 + 多池子做事件分析

链上每笔 swap 都会触发 `Swap` 事件，你可以扫描得到完整交易历史。

```python
# 拿最近 1000 个块内的 Swap 事件
from_block = w3.eth.block_number - 1000
events = pair.events.Swap().get_logs(fromBlock=from_block)

for ev in events[:5]:
    a = ev.args
    print(f"Block {ev.blockNumber}: ", end="")
    print(f"in0={a.amount0In/10**6:.2f} in1={a.amount1In/10**18:.4f} "
          f"out0={a.amount0Out/10**6:.2f} out1={a.amount1Out/10**18:.4f}")
```

这就是**链上量化的数据源**。所有的"巨鲸地址监控"、"聪明钱跟单"、"MEV 检测"都基于这种数据。

## 7. 链上数据三大法宝：node、The Graph、Dune

你自己扫合约事件太慢。专业人士用：

### 7.1 自建 / 租用 Archive Node

**Archive 节点**保存完整链历史（>= 15TB），可以查任何时刻的状态。

| 服务 | 类型 | 价格 |
|---|---|---|
| Alchemy | 托管 | 免费 1.5M CU/月，付费起 $49/月 |
| Infura | 托管 | 类似 Alchemy |
| QuickNode | 托管 | 类似 |
| Erigon | 自建 | 硬件成本 $5k+ |
| Reth (Rust) | 自建 | 类似 Erigon |

**入门用 Alchemy 免费版就够。** 真实生产建议自建（不会被节流、隐私好）。

### 7.2 The Graph：链上数据的"GraphQL"

The Graph 把链上事件**索引成 GraphQL 数据库**。

举例：查 Uniswap V3 最近 10 个最大池子：

```graphql
{
  pools(first: 10, orderBy: totalValueLockedUSD, orderDirection: desc) {
    id
    token0 { symbol }
    token1 { symbol }
    totalValueLockedUSD
    volumeUSD
  }
}
```

每个主流协议都有自己的 subgraph。**比扫 Etherscan 快 100 倍**。

### 7.3 Dune Analytics：SQL 查链上数据

[Dune](https://dune.com) 是链上数据分析的"BI 平台"。

- 用 SQL 查所有主流公链（ETH、BTC、Solana、L2）
- 社区已经写好几万个 dashboard 可以复用
- 适合做"研究报告"，不适合实时高频应用

**经典 SQL 例子**：查 Uniswap V3 各池子今日交易量：

```sql
SELECT
    pool,
    SUM(amount_usd) AS volume_usd_today,
    COUNT(*) AS n_trades
FROM uniswap_v3_ethereum.trades
WHERE evt_block_time >= NOW() - INTERVAL '1 day'
GROUP BY pool
ORDER BY volume_usd_today DESC
LIMIT 20;
```

**应聘 DeFi 数据分析师，Dune 是必会工具。**

## 8. 你必须了解的 DeFi 协议

| 协议 | 类型 | TVL（峰值）| 必读理由 |
|---|---|---|---|
| **Uniswap V2/V3** | DEX | $10B+ | AMM 鼻祖，所有人都集成它 |
| **Curve** | DEX | $25B+ | 稳定币 AMM，独特的 StableSwap |
| **Aave** | 借贷 | $15B+ | 借贷标准，闪电贷起源地 |
| **Compound** | 借贷 | $12B+ | Aave 的前辈，治理代币模式开创者 |
| **MakerDAO** | 稳定币 | $20B+ | DAI 发行方，超额抵押稳定币始祖 |
| **Lido** | 流动质押 | $25B+ | ETH 质押凭证 stETH 是 DeFi 重要资产 |
| **GMX** | 永续合约 | $1B+ | DEX 永续合约领导者 |
| **dYdX** | 永续合约 | $1B+ | 订单簿型 DEX 代表 |
| **Convex** | Yield 聚合 | $5B+ | Curve 的 boost 收益玩法 |
| **Yearn** | Yield 聚合 | $4B+ | 自动复投鼻祖 |

**每读一个，问自己**：
1. 它解决什么问题？传统金融里有什么对应物？
2. 它的代币经济模型是什么？token 拿来干嘛？
3. 它最可能被哪种攻击？历史上发生过什么事件？

### 学习顺序（推荐）

1. **Uniswap V2** → 理解 AMM
2. **Compound** → 理解借贷
3. **MakerDAO** → 理解稳定币
4. **Curve** → 理解 StableSwap、Bonding Curve、ve 模型
5. **Uniswap V3** → 理解集中流动性
6. **Aave V3** → 理解闪电贷、eMode、隔离资产
7. **GMX / dYdX** → 理解永续合约

## 9. 你应该做的 3 个链上数据分析项目

学完上面的，做这 3 个项目能让简历亮眼：

### 项目 1：DEX 价差监控

- 扫描 Uniswap V2、SushiSwap、Curve 上同一对的价格
- 找出价差最大的池子对
- 计算扣除 gas 后的理论套利空间
- 用 Dune dashboard 展示

**学到**：链上 RPC、事件解析、价差套利逻辑

### 项目 2：聪明钱（Smart Money）追踪

- 选 50 个历史交易胜率高的地址（来源：Nansen / 自己挖）
- 实时监控这些地址的交易
- 当 N 个智能钱包同时买入某 token 时报警
- 在 Discord / Telegram 发送提醒

**学到**：地址画像、信号设计、实时数据流

### 项目 3：DeFi 协议收益分析

- 跑一遍 Aave、Compound、Curve、Yearn 各池子收益率
- 计算"风险调整后收益"（考虑各协议历史 exploit 风险）
- 输出一个"DeFi 投资组合"建议
- 写成 medium 文章发布

**学到**：协议机制、收益对比、风险评估

## 10. 本节小结

**你现在应该会的：**

- ✅ 理解 AMM 恒定乘积、滑点、无常损失
- ✅ Python 实现一个 mini Uniswap V2
- ✅ 理解借贷协议的利率曲线、清算、闪电贷
- ✅ 用 web3.py 读真实主网状态
- ✅ 知道 The Graph、Dune 的用法和适用场景

---

**下一节：`07_链上量化_MEV与套利.ipynb`**

我们会讲：
- 跨 DEX 套利的代码实现
- 三明治攻击（sandwich）原理
- MEV bot 架构
- Flashbots、PBS、Builder 生态
- **链上量化 / Crypto Quant 的求职路径**（GSR、Wintermute、Jump Crypto 怎么进）